In [1]:
import numpy as np
import pandas as pd
import rdkit 
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import rdMolDraw2D
from matplotlib import cm
from matplotlib import pyplot as plt
from mordred import Calculator, descriptors
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# 記述子計算用インスタンスを用意
calc = Calculator(descriptors, ignore_3D = True)

In [3]:
df = pd.read_csv('../../kampo_db/kampo_data_original/230922/check/kampo_ESI_tateishi_1004_checked.csv')
train = pd.read_csv('231202_tateishi_a-glucosidase_ECFP4.csv')
all_variables =train.columns.values

df_smiles = df['smiles']
mols_list = []
for i in df_smiles:
    mol = Chem.MolFromSmiles(i)
    mols_list.append(mol)
    
# ECFP4フィンガープリントの作成（Extended Connectivity Fingerprint）
ECFP4 = [AllChem.GetMorganFingerprintAsBitVect(x,2,2048) for x in mols_list]
ECFP4 = pd.DataFrame(np.array(ECFP4, int))
ECFP4.dropna(axis=1,inplace=True)
e = ECFP4.columns.values
type(e[2])

numpy.int64

In [4]:
variables = all_variables[6:].tolist()
variables = [int(n) for n in variables]
type(variables[2])

int

In [5]:
len(variables)

839

In [6]:
"""threshold = 0.8
feat_corr = set()
corr_matrix = ECFP4.corr()
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > threshold:
            feat_name = corr_matrix.columns[i]
            feat_corr.add(feat_name)"""

'threshold = 0.8\nfeat_corr = set()\ncorr_matrix = ECFP4.corr()\nfor i in range(len(corr_matrix.columns)):\n    for j in range(i):\n        if abs(corr_matrix.iloc[i, j]) > threshold:\n            feat_name = corr_matrix.columns[i]\n            feat_corr.add(feat_name)'

In [7]:
""""print(f'削除分{len(set(feat_corr))}')

ECFP4.drop(labels=feat_corr, axis=1, inplace=True)
ECFP4.dropna(how="any", axis=1, inplace=True)
print(f'残り{len(ECFP4.columns)}')

ECFP4_columns = ECFP4.columns.values
len(ECFP4_columns)""""

SyntaxError: unterminated string literal (detected at line 8) (1675971057.py, line 8)

In [8]:
ECFP4[variables]

,1,9,10,11,13,14,16,18,19,25,...,2012,2015,2021,2024,2030,2032,2033,2034,2041,2042
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2644,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2645,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2646,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2647,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
#ECFP4_train = pd.concat([df1.iloc[:352,:6], ECFP4[:352]], axis=1)
#ECFP4_test = pd.concat([df2.iloc[:89,:6], ECFP4[352:].reset_index()], axis=1)

In [7]:
#ECFP4_test

ECFP4 = pd.concat([df[["number", "cid", "name", "smiles"]], ECFP4[variables]], axis=1)
ECFP4 = ECFP4.dropna(how='any')
for i in ECFP4.index:
    if (ECFP4.loc[i] == "inf").any():
        ECFP4.drop(i, axis=0,inplace=True)
    if (ECFP4.loc[i] == "nan").any():
        ECFP4.drop(i, axis=0,inplace=True)

ECFP4

,number,cid,name,smiles,1,9,10,11,13,14,...,2012,2015,2021,2024,2030,2032,2033,2034,2041,2042
0,kampo_0002,11095734,(+)-Aromadendrene,CC1CCC2C1C3C(C3(C)C)CCC2=C,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,kampo_0003,73533,(+)-Catechin 7-O-beta-D-xyloside,C1C(C(OC2=CC(=CC(=C21)O)OC3C(C(C(CO3)O)O)O)C4=...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,kampo_0004,167812,(+)-Curcumenol,CC1CCC2C13CC(=C(C)C)C(O3)(C=C2C)O,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,kampo_0005,16212927,(+)-Cyclosativene,CC(C)C1CCC2(C3C1C4C2(C4C3)C)C,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,kampo_0006,182232,(+)-Epicatechin,C1C(C(OC2=CC(=CC(=C21)O)O)C3=CC(=C(C=C3)O)O)O,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2644,kampo_2996,5282769,trans-11-Eicosenoic acid,CCCCCCCCC=CCCCCCCCCCC(=O)O,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2645,kampo_2997,639662,trans-2-Heptene,CCCCC=CC,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2646,kampo_2998,5284503,trans-3-Hexen-1-ol,CCC=CCCO,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2647,kampo_3001,88302,trans-Pinocarveol,CC1(C2CC1C(=C)C(C2)O)C,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
ECFP4.to_csv('231207_tateishi_ECFP4_kampo.csv', index=False)

In [21]:
#保存
#ECFP4_train.to_csv('230125_tateishi_a-glucosidase_ECFP4_train.csv', index=False)
#ECFP4_test.to_csv('230125_tateishi_a-glucosidase_ECFP4_test.csv', index=False)